In [3]:
from Bio import SeqIO

record = SeqIO.read("inzert.fasta", "fasta")

print(record.id)
print(record.description)
print(len(record.seq))
print(record.seq[:30])

MZ733680.1
MZ733680.1 Bacillus sp. (in: Bacteria) strain T6 xylanase (xynA) gene, complete cds
642
ATGTTTAAGTTTAAAAAGAATTTCTTAGTT


In [4]:
from Bio import SeqIO
from Bio.Seq import Seq

# 1. Load your local fasta file
record = SeqIO.read("inzert.fasta", "fasta")

print(f"File ID: {record.id}")
print(f"Total DNA Length: {len(record.seq)} bp")
print(f"First 30 DNA bases: {record.seq[:30]}")

# 2. Translate the nucleotide sequence to an Amino Acid sequence
# (E. coli expression requires working at the protein level)
protein_seq = record.seq.translate()
print(f"Translated Protein Length: {len(protein_seq)} aa")
print(f"First 10 Amino Acids: {protein_seq[:10]}")

# 3. Apply standard bacterial Signal Peptide rules
# Bacterial signal peptides are typically 15-30 amino acids long,
# starting with a positively charged region (K, R) and a hydrophobic core (A, L, V, I, F).
def find_signal_peptide_boundary(amino_acids):
    # Standard fallback cutoff if no structural match is found
    default_cutoff = 25 
    
    # Locate the end of the initial highly hydrophobic core region
    # Look for the transition back to hydrophilic/polar regions (D, E, K, R, Q, N)
    for i in range(12, min(40, len(amino_acids))):
        sub_seq = amino_acids[i:i+5]
        # Count polar residues in a 5-amino-acid sliding window
        polar_count = sum(1 for aa in sub_seq if aa in "DEKRHQNST")
        if polar_count >= 3:
            return i + 2 # Estimated cleavage location site
            
    return default_cutoff

# Calculate positions
aa_cutoff = find_signal_peptide_boundary(protein_seq)
dna_cutoff = aa_cutoff * 3

print("\n" + "="*40)
print(" WET-LAB PROCESSING INSTRUCTIONS ")
print("="*40)
print(f"• Estimated Cleavage Site: After Amino Acid position {aa_cutoff}")
print(f"• DNA Boundary Location  : Base 0 to Base {dna_cutoff}")
print(f"• Mature Peptide Sequence: {record.seq[dna_cutoff:dna_cutoff+30]}...")
print("-"*40)
print(f"• ACTION FOR BENCHLING: Delete bases 1 to {dna_cutoff}.")
print("  Then insert 'ATG' manually at the start to allow E. coli translation.")
print("="*40)


File ID: MZ733680.1
Total DNA Length: 642 bp
First 30 DNA bases: ATGTTTAAGTTTAAAAAGAATTTCTTAGTT
Translated Protein Length: 214 aa
First 10 Amino Acids: MFKFKKNFLV

 WET-LAB PROCESSING INSTRUCTIONS 
• Estimated Cleavage Site: After Amino Acid position 24
• DNA Boundary Location  : Base 0 to Base 72
• Mature Peptide Sequence: ACCGCCTCTGCAGCTAGTCCAGACTACTGG...
----------------------------------------
• ACTION FOR BENCHLING: Delete bases 1 to 72.
  Then insert 'ATG' manually at the start to allow E. coli translation.


In [5]:
from Bio import SeqIO
from Bio.Seq import Seq

# =====================================================================
# STEP 1: LOAD GENE AND VECTOR SEQUENCES
# =====================================================================
# Load your xylanase file
gene_record = SeqIO.read("inzert.fasta", "fasta")
mature_xylanase = gene_record.seq

# Define your empty pET28a sequence (simplified representation of the MCS area)
# In your real workflow, load it via SeqIO if you have the pET28a.fasta file
pet28a_seq = Seq("TGGCGAATGGGACGCGCCCTGTAGCATATGGCTAGCATGACTGGTGGACAGCAAATGGGTCGCGGATCCGAATTC")
# Note: 'CATATG' is the NdeI restriction site located inside the pET28a MCS

print(f"Original Gene Length: {len(mature_xylanase)} bp")
print(f"Vector Length: {len(pet28a_seq)} bp")

# =====================================================================
# STEP 2: FIND WHERE TO CUT THE DNA (RESTRICTION SITE)
# =====================================================================
# We will use NdeI (CATATG) to align and cut the DNA
restriction_site = "CATATG"

# Find where the enzyme cuts the vector
vector_cut_index = pet28a_seq.find(restriction_site)
if vector_cut_index == -1:
    raise ValueError("Restriction site not found in vector!")

print(f"\n[CUTTING MAP]")
print(f"• Cutting Vector at index position: {vector_cut_index} ({restriction_site})")

# =====================================================================
# STEP 3: VIRTUAL INSERTION (CLONING BY BASE EDITING)
# =====================================================================
# We split the vector open at the restriction site and place the gene inside
vector_left_arm = pet28a_seq[:vector_cut_index]
vector_right_arm = pet28a_seq[vector_cut_index + len(restriction_site):]

# Combine the pieces to form the recombinant plasmid sequence
recombinant_plasmid = vector_left_arm + restriction_site + mature_xylanase + vector_right_arm
print(f"\n[CLONING SUCCESSFUL]")
print(f"• New Recombinant Plasmid Length: {len(recombinant_plasmid)} bp")

# =====================================================================
# STEP 4: MUTATING SPECIFIC BASES IN THE ENZYME
# =====================================================================
# Strings are immutable in Python, so we convert the sequence to a mutable list
mutable_plasmid = list(str(recombinant_plasmid))

# Example: We want to introduce a point mutation to improve xylanase activity.
# Let's say we want to mutate base index position 120 from an 'A' to a 'G'.
mutation_target_index = vector_cut_index + 120  # Offset by the vector arm length
original_base = mutable_plasmid[mutation_target_index]

# Swap the base
mutable_plasmid[mutation_target_index] = "G" 
mutated_plasmid_seq = Seq("".join(mutable_plasmid))

print(f"\n[MUTAGENESIS SUMMARY]")
print(f"• Targeted global index position: {mutation_target_index}")
print(f"• Changed base character from [{original_base}] to [G]")
print(f"• Verified Mutation Region: ...{mutated_plasmid_seq[mutation_target_index-5:mutation_target_index+6]}...")

# =====================================================================
# STEP 5: EXPORT COMPLETED CONSTRUCT BACK TO BENCHLING
# =====================================================================
with open("recombinant_mutated_xylanase.fasta", "w") as output_file:
    output_file.write(f">pET28a_Xylanase_Mutated\n{mutated_plasmid_seq}\n")
print("\n🎉 Process complete! File saved as 'recombinant_mutated_xylanase.fasta'.")
print("You can now import this file directly back into your Benchling workspace.")


Original Gene Length: 642 bp
Vector Length: 75 bp

[CUTTING MAP]
• Cutting Vector at index position: 24 (CATATG)

[CLONING SUCCESSFUL]
• New Recombinant Plasmid Length: 717 bp

[MUTAGENESIS SUMMARY]
• Targeted global index position: 144
• Changed base character from [G] to [G]
• Verified Mutation Region: ...GGACTGATGGC...

🎉 Process complete! File saved as 'recombinant_mutated_xylanase.fasta'.
You can now import this file directly back into your Benchling workspace.
